### This notebook goes over the various ways to fix our ACE data and uses TPOT to generate a pipeline for ML.

In [21]:
#add our library to our python path
import sys
sys.path.append('/home/cbo27/gbcompare')
import fix_functions as ff

#Library Imports
import pickle
import os
import glob
import numpy as np
import scipy.linalg.interpolative as sli

As stated in the readme there are 4 ways to standardize our data across the many gb systems.
1. Average Row Representation (ARR)
2. Kernel-PCA Kmeans
3. Skeleton Decomposition
4. Largest Simplex

For our example we are going to use the ARR.

If you desire to use any of the other methods follow the intructions below.

### Kernel-PCA Kmeans
Replace:
```
    fixed_data = np.average(data, axis=0)
```
With:
```
    fixed_data = ff.kernel_pca_kmeans(data,10,100)
```

### Skeleton Decomposition
Replace:
```
    fixed_data = np.average(data, axis=0)
    ...
    with open('fixed_data/' + new_filename, 'wb') as f:
        pickle.dump(fixed_data, f)
```
With:
```
    idx, proj = sli.interp_decomp(np.transpose(data), 20)
    fixed_data = sli.reconstruct_skel_matrix(np.transpose(data), 20, idx)
    ...
    with open('fixed_data/' + new_filename, 'wb') as f:
        pickle.dump(np.transpose(fixed_data), f)
```

### Largest Simplex
Replace:
```
    fixed_data = np.average(data, axis=0)
```
With:
```
    my_matrix = ff.find_largest_simplex(data,10)
```

In [19]:
# define the directory
dir_path = 'ace_txt_data'
start_indx = len(dir_path) + 1
# iterate through each pickle file in the directory
for filename in glob.glob(os.path.join(dir_path, '*.txt')):

    # load the numpy array from the pickle file
    data = np.loadtxt(filename, converters=float)
    
    # calculate the average with axis=0
    fixed_data = np.average(data, axis=0)

    # create the new filename
    new_filename = os.path.splitext(filename[start_indx:])[0] + '_fixed.pickle'
    
    directory = 'fixed_data'

    if not os.path.exists(directory):
        os.makedirs(directory)
        print("Directory created:", directory)
    else:
        print("Directory already exists:", directory)
    
    #write the average array to the new pickle file
    with open('fixed_data/' + new_filename, 'wb') as f:
        pickle.dump(np.transpose(fixed_data), f)

### The next part of this notebook is taking our new grain boundary descriptor and using topt to find an optimal ML pipline.

NOTE!!! IF you are using any method other than ARR you need to replace the line:
```
X = np.array(x_list)
```
with:
```
X = np.array(x_list).reshape(7174,-1)
```
This is because sklearn needs 1D data for inputs

sample_energy.npy is a numpy array storing the labels for each grain boundary. It is ordered alphabetically by the compute_id. 

In [20]:
from sklearn.pipeline import make_pipeline, make_union
from tpot import TPOTRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer
from sklearn.metrics import mean_absolute_error

path = 'fixed_data'
my_list_dir = np.sort(os.listdir(path))

x_list = []
for file in my_list_dir:
    with open(path+'/'+file,'rb') as f:
        amr = pickle.load(f)
        x_list.append(amr)

#use this line if using anything other than ARR
#X = np.array(x_list).reshape(7174,-1)
X = np.array(x_list)
y = np.load('../sample_energy.npy')

X_train, X_test, y_train, y_test = train_test_split(X, y,train_size=0.75, test_size=0.25)

my_custom_scorer = make_scorer(mean_absolute_error, greater_is_better=False)
tpot = TPOTRegressor(generations=5, population_size=50,verbosity=2,scoring=my_custom_scorer)
tpot.fit(X_train, y_train)
print(tpot.score(X_test, y_test))

                                                                             
Generation 1 - Current best internal CV score: -22.45599975759623
                                                                              
Generation 2 - Current best internal CV score: -22.45599975759622
                                                                              
Generation 3 - Current best internal CV score: -22.45599975759622
                                                                              
Generation 4 - Current best internal CV score: -22.45599975759622
                                                                              
Generation 5 - Current best internal CV score: -22.45599975759622
                                                                              
Best pipeline: ElasticNetCV(MaxAbsScaler(XGBRegressor(input_matrix, learning_rate=0.5, max_depth=4, min_child_weight=7, n_estimators=100, n_jobs=1, objective=reg:squarederror, subsample=0.3, verbos